In [ ]:
# --- CELL 1: SETUP & CONFIG ---
import os
import sys
import json
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import BertModel, BertTokenizerFast
import re

# Install dependencies (if running on fresh Kaggle environment)
!pip install transformers --quiet

# --- PATH CONFIGURATION ---
# Adjust these paths based on where your data is located in Kaggle
BASE_INPUT_DIR = "/kaggle/input"

# Assuming you uploaded the JSON and txt files as datasets
DRUG_JSON_PATH = os.path.join(BASE_INPUT_DIR, "decoder-slm/pytorch/default/1/Drug_Slang_Data/drugNames.json")
TRAIN_TXT_PATH = os.path.join(BASE_INPUT_DIR, "train-test/train_data.txt")
TEST_TXT_PATH  = os.path.join(BASE_INPUT_DIR, "train-test/test_data.txt")

MODEL_SAVE_PATH = "property_model.pt"

# Check if CUDA (GPU) is available
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Hyperparameters
BATCH_SIZE = 32
EPOCHS = 3
LEARNING_RATE = 2e-5
TRANSFORM_THRESHOLD = 0.50
CONTEXT_THRESHOLD = 0.80

# Fallback/Mock paths for testing if files don't exist
if not os.path.exists(DRUG_JSON_PATH):
    print("⚠️ Warning: Dataset paths not found. Please double check paths in Cell 1.")

In [ ]:
# --- CELL 2: DATA LOADER & UTILS ---

def load_drug_map(path):
    """Load drug -> slang mapping from JSON."""
    try:
        with open(path, 'r', encoding='utf-8') as f:
            content = f.read().replace('\n', ' ')
            data = json.loads(content)
        
        # Create reverse mapping: slang -> drug
        slang_map = {}
        for drug, slangs in data.items():
            for slang in slangs:
                slang_map[slang.lower()] = drug
        return slang_map
    except Exception as e:
        print(f"Error loading drug map: {e}")
        return {}

def load_samples(path):
    """Load samples from text file (sentence|slang|drug|label)."""
    samples = []
    if not os.path.exists(path):
        print(f"File not found: {path}")
        return []
        
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            parts = line.split('|')
            if len(parts) == 4:
                samples.append(tuple(parts))
    return samples

class PropertyDataset(Dataset):
    """Dataset for training property heads."""
    def __init__(self, samples, tokenizer, max_len=64):
        self.samples = samples
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sentence, slang_word, drug_name, label = self.samples[idx]
        
        # Tokenize sentence
        sent_enc = self.tokenizer(sentence, return_tensors="pt", truncation=True, 
                                  max_length=self.max_len, padding="max_length")
        
        # Tokenize slang word (for word property)
        word_enc = self.tokenizer(slang_word, return_tensors="pt", truncation=True,
                                  max_length=32, padding="max_length")
        
        # Tokenize drug name (for drug property)
        drug_enc = self.tokenizer(drug_name, return_tensors="pt", truncation=True,
                                  max_length=32, padding="max_length")
        
        # Label: 1 for drug context, -1 for innocent context
        y = 1 if label == "drug_context" else -1
        
        return {
            "sent_ids": sent_enc["input_ids"].squeeze(0),
            "sent_mask": sent_enc["attention_mask"].squeeze(0),
            "word_ids": word_enc["input_ids"].squeeze(0),
            "word_mask": word_enc["attention_mask"].squeeze(0),
            "drug_ids": drug_enc["input_ids"].squeeze(0),
            "drug_mask": drug_enc["attention_mask"].squeeze(0),
            "label": y
        }

# Load Global Data
try:
    tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
    slang_map = load_drug_map(DRUG_JSON_PATH)
    train_samples_list = load_samples(TRAIN_TXT_PATH)
    test_samples_list = load_samples(TEST_TXT_PATH)
    print(f"Loaded {len(slang_map)} slang terms.")
    print(f"Loaded {len(train_samples_list)} training samples.")
    print(f"Loaded {len(test_samples_list)} test samples.")
except Exception as e:
    print(f"Initialization error (check paths): {e}")

In [ ]:
# --- CELL 3: MODEL & TRAINING ---

class PropertyModel(nn.Module):
    """
    BERT-based model with two property heads:
    1. Word-property head: extracts innocent semantics
    2. Drug-property head: extracts criminal-intent semantics
    """
    def __init__(self, bert_name="bert-base-uncased", hidden_dim=768):
        super().__init__()
        self.bert = BertModel.from_pretrained(bert_name)
        
        # Word property head 
        self.word_property_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        
        # Drug property head
        self.drug_property_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
    
    def forward(self, word_ids=None, word_mask=None, drug_ids=None, drug_mask=None, input_ids=None, attention_mask=None):
        # Handle sentence embedding check (pure BERT)
        if input_ids is not None:
             out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
             return out

        # Training pass
        word_out = self.bert(input_ids=word_ids, attention_mask=word_mask)
        drug_out = self.bert(input_ids=drug_ids, attention_mask=drug_mask)
        
        word_emb = word_out.last_hidden_state[:, 0, :] 
        drug_emb = drug_out.last_hidden_state[:, 0, :] 
        
        word_property = self.word_property_head(word_emb)
        drug_property = self.drug_property_head(drug_emb)
        
        return F.normalize(word_emb, p=2, dim=1), F.normalize(word_property, p=2, dim=1), \
               F.normalize(drug_emb, p=2, dim=1), F.normalize(drug_property, p=2, dim=1)

def train_model():
    if not train_samples_list:
        print("No training data found. Skipping training.")
        return

    dataset = PropertyDataset(train_samples_list, tokenizer)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    
    model = PropertyModel().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CosineEmbeddingLoss(margin=0.3)
    
    print(f"Started Training on {len(dataset)} samples for {EPOCHS} epochs...")
    
    model.train()
    for epoch in range(EPOCHS):
        total_loss = 0
        for i, batch in enumerate(dataloader):
            word_ids = batch["word_ids"].to(DEVICE)
            word_mask = batch["word_mask"].to(DEVICE)
            drug_ids = batch["drug_ids"].to(DEVICE)
            drug_mask = batch["drug_mask"].to(DEVICE)
            labels = batch["label"].float().to(DEVICE)
            
            optimizer.zero_grad()
            
            w_emb, w_prop, d_emb, d_prop = model(word_ids=word_ids, word_mask=word_mask, 
                                                 drug_ids=drug_ids, drug_mask=drug_mask)
            
            # Vector Alchemy: V_transformed = V_word - V_prop + V_drug_prop
            transformed = F.normalize(w_emb - w_prop + d_prop, p=2, dim=1)
            
            loss = criterion(transformed, d_emb, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
            if i % 50 == 0:
                print(f"Epoch {epoch+1} | Batch {i} | Loss: {loss.item():.4f}")
        
        print(f"Epoch {epoch+1} Complete. Avg Loss: {total_loss/len(dataloader):.4f}")
        
    torch.save({
        'model_state_dict': model.state_dict()
    }, MODEL_SAVE_PATH)
    print(f"Model saved to {MODEL_SAVE_PATH}")
    return model

# Run Training
model = train_model()

In [ ]:
# --- CELL 4: DETECTIVE / INFERENCE ---

def compute_sentence_embedding(model, sentence):
    enc = tokenizer(sentence, return_tensors='pt', truncation=True, max_length=64).to(DEVICE)
    with torch.no_grad():
        out = model(input_ids=enc['input_ids'], attention_mask=enc['attention_mask'])
    return F.normalize(out.last_hidden_state[:, 0, :].squeeze(0), p=2, dim=0)

def detect_slang(model, sentence):
    model.eval()
    detections = []
    sentence_lower = sentence.lower()
    
    # 1. Candidate Generation via Phrase Matching
    sorted_slang_keys = sorted(slang_map.keys(), key=len, reverse=True)
    candidates = []
    for slang in sorted_slang_keys:
        if len(slang) < 3: continue # Skip tiny words to reduce noise
        pattern = r'\b' + re.escape(slang) + r'\b'
        if re.search(pattern, sentence_lower):
            candidates.append(slang)

    if not candidates:
        return []

    try:
        # Get original sentence embedding
        sent_emb_original = compute_sentence_embedding(model, sentence)
        
        for slang_phrase in candidates:
            drug_name = slang_map[slang_phrase]
            
            # Replace phrase with drug name
            pattern = re.compile(r'\b' + re.escape(slang_phrase) + r'\b', re.IGNORECASE)
            sentence_replaced = pattern.sub(drug_name, sentence)
            
            sent_emb_replaced = compute_sentence_embedding(model, sentence_replaced)
            
            # Get vectors
            word_enc = tokenizer(slang_phrase, return_tensors='pt', max_length=32, padding="max_length", truncation=True).to(DEVICE)
            drug_enc = tokenizer(drug_name, return_tensors='pt', max_length=32, padding="max_length", truncation=True).to(DEVICE)
            
            with torch.no_grad():
                w_emb, w_prop, d_emb, d_prop = model(
                    word_ids=word_enc['input_ids'], word_mask=word_enc['attention_mask'],
                    drug_ids=drug_enc['input_ids'], drug_mask=drug_enc['attention_mask']
                )
                
                w_emb, w_prop, d_emb, d_prop = w_emb.squeeze(0), w_prop.squeeze(0), d_emb.squeeze(0), d_prop.squeeze(0)
            
            # Vector Alchemy
            transformed = F.normalize(w_emb - w_prop + d_prop, p=2, dim=0)
            
            # Metrics
            sim_transform_drug = F.cosine_similarity(transformed.unsqueeze(0), d_emb.unsqueeze(0)).item()
            sim_context = F.cosine_similarity(sent_emb_original.unsqueeze(0), sent_emb_replaced.unsqueeze(0)).item()
            
            if sim_transform_drug >= TRANSFORM_THRESHOLD and sim_context >= CONTEXT_THRESHOLD:
                detections.append((slang_phrase, drug_name, sim_transform_drug, sim_context))
                
    except Exception as e:
        print(f"Detection Error: {e}")

    return detections

def evaluate_test_set(model):
    if not test_samples_list:
        print("No test data to evaluate.")
        return

    print("Evaluating on Test Set...")
    tp, fp, fn, tn = 0, 0, 0, 0
    total = 0
    
    for entry in test_samples_list:
        sentence, target_slang, drug, label = entry
        total += 1
        
        # Run Detection
        results = detect_slang(model, sentence)
        detected_slangs = [r[0].lower() for r in results]
        is_detected = target_slang.lower() in detected_slangs
        is_drug_context = (label == "drug_context")
        
        if is_drug_context and is_detected: tp += 1
        elif is_drug_context and not is_detected: fn += 1
        elif not is_drug_context and is_detected: fp += 1
        else: tn += 1
        
        if total % 200 == 0: print(f"Processed {total}...")

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    print(f"\n--- Results ---")
    print(f"TP: {tp} | FP: {fp} | FN: {fn} | TN: {tn}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")

# Run Evaluation
if model:
    evaluate_test_set(model)

# Test Examples
test_sentences = [
    "I need some snow for the party tonight",
    "The snow is beautiful outside",
    "Looking to buy some ice",
    "Put some ice in my drink"
]
print("\n--- Visual Sanity Check ---")
for s in test_sentences:
    res = detect_slang(model, s)
    if res:
        print(f"⚠️  {s} -> {res}")
    else:
        print(f"✓ {s} -> Innocent")